# 01. BMIQ Normalization — (GSE287331)

This *R-script* performs full **Type-I/Type-II bias correction** on the GSE287331 EPIC dataset. After loading the β-matrix and phenotype metadata from Parquet, it imports the official Illumina EPIC manifest and constructs a CpG-wise Infinium design vector aligned to the matrix columns. A memory-lean sequential BMIQ procedure is applied sample-by-sample, using robust probe-type checks to safely handle incomplete design annotations. The corrected β-values are reconstructed together with *id_tissue* and *label* and saved as a compressed LZ4 Parquet file. The resulting dataset is fully normalized, probe-type balanced, and ready for cross-dataset comparability and downstream analysis.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     01-correction-bias-type-i-vs-type-ii               ║
# ║ Description:  BMIQ normalization correcting Infinium Type I/II   ║
# ║               probe-design bias in the GSE287331 EPIC dataset.   ║
# ║ Dataset(s):   GSE287331                                          ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 17-Nov-2025 | Language: R 4.4.0                            ║
# ╚══════════════════════════════════════════════════════════════════╝


## BMIQ Correction

In [ ]:
# BMIQ CORRECTION FOR GSE287331 (SEQUENTIAL, MEMORY-LEAN)
# 1. Install and load required packages 
# 1.1 Define required CRAN packages
needed_cran <- c("arrow", "data.table")

# 1.2 Install missing CRAN packages (if any)
for (pkg in needed_cran) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
  }
}

# 1.3 Install BiocManager if missing
if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}

# 1.4 Install wateRmelon from Bioconductor if missing
if (!requireNamespace("wateRmelon", quietly = TRUE)) {
  BiocManager::install("wateRmelon")
}

# 1.5 Load libraries
library(arrow)
library(data.table)
library(wateRmelon)

# 2. Settings 
# 2.1 Paths for input β-matrix and output corrected Parquet
INPUT_PARQUET  <- "/kaggle/input/3-gse287331-parquet/GSE287331_clean_imputed.parquet"
OUTPUT_PARQUET <- "/kaggle/working/GSE287331_BMIQ.parquet"

# 2.2 Path to Illumina EPIC manifest 
MANIFEST_PATH  <- "/kaggle/input/manifest-infinium-methylationepic-cpg-to-gene/infinium-methylationepic-v-1-0-b5-manifest-file.csv"

# 2.3 Column names for sample ID and label in the Parquet file
ID_COL    <- "id_tissue"
LABEL_COL <- "label"

# 3. Helper functions for robust manifest loading 
# 3.1 Detect the field separator from a header line
detect_separator <- function(sample_line, default = ",") {
  if (grepl("\t", sample_line, fixed = TRUE)) return("\t")
  if (grepl(";",  sample_line, fixed = TRUE)) return(";")
  if (grepl(",",  sample_line, fixed = TRUE)) return(",")
  return(default)
}

# 3.2 Open a text connection, supporting plain, .gz and .bz2 files
open_text_auto <- function(path) {
  if (grepl("\\.gz$", path)) {
    con <- gzfile(path, open = "rt")
  } else if (grepl("\\.bz2$", path)) {
    con <- bzfile(path, open = "rt")
  } else {
    con <- file(path, open = "rt")
  }
  con
}

# 3.3 Load Illumina manifest and normalise the CpG ID column to 'CpG'
load_illumina_manifest_any <- function(path,
                                       header_probe_keys = c("IlmnID", "Probe_ID", "Name",
                                                             "TargetID", "ID_REF", "CpG"),
                                       fallback_skiprows = 6L) {
  if (!file.exists(path)) {
    stop("[ERROR] Manifest file does not exist: ", path)
  }
  
  con <- open_text_auto(path)
  on.exit(close(con), add = TRUE)
  
  lines <- readLines(con, warn = FALSE)
  if (length(lines) < 8L) {
    stop("[ERROR] Manifest file too short to contain header + data.")
  }
  
  header_idx <- NA_integer_
  
  # 3.3.1 First, look for the [Assay] section and take the next non-empty line as header
  max_scan <- min(200L, length(lines))
  for (i in seq_len(max_scan)) {
    if (startsWith(tolower(trimws(lines[i])), "[assay]")) {
      upper_j <- min(i + 50L, length(lines))
      for (j in (i + 1L):upper_j) {
        s <- trimws(lines[j])
        if (nchar(s) > 0L && !startsWith(s, "[")) {
          header_idx <- j
          break
        }
      }
      if (!is.na(header_idx)) break
    }
  }
  
  # 3.3.2 Fallback: search for a line that looks like a header (contains known CpG ID keys)
  if (is.na(header_idx)) {
    max_scan2 <- min(50L, length(lines))
    for (i in seq_len(max_scan2)) {
      s <- trimws(lines[i])
      parts <- strsplit(s, ",", fixed = TRUE)[[1]]
      if (length(parts) == 1L) {
        parts <- strsplit(s, "\t", fixed = TRUE)[[1]]
      }
      if (length(parts) == 1L) {
        parts <- strsplit(s, ";", fixed = TRUE)[[1]]
      }
      parts <- trimws(parts)
      if (any(parts %in% header_probe_keys)) {
        header_idx <- i
        break
      }
    }
  }
  
  # 3.3.3 Ultimate fallback if nothing was detected
  if (is.na(header_idx)) {
    header_idx <- fallback_skiprows + 1L
    warning("[manifest] Header not detected → applying fallback skiprows = ", fallback_skiprows)
  }
  
  header_line <- lines[header_idx]
  sep <- detect_separator(header_line, default = ",")
  
  # 3.3.4 Build a text block starting from the header line
  csv_text <- paste(lines[header_idx:length(lines)], collapse = "\n")
  
  man <- data.table::fread(
    input        = csv_text,
    sep          = sep,
    header       = TRUE,
    data.table   = TRUE,
    showProgress = FALSE
  )
  
  # 3.3.5 Normalise CpG ID column name to 'CpG'
  cpg_col <- NULL
  for (cand in header_probe_keys) {
    if (cand %in% names(man)) {
      data.table::setnames(man, cand, "CpG")
      cpg_col <- "CpG"
      break
    }
  }
  if (is.null(cpg_col)) {
    stop("[ERROR] 'CpG' column not found in manifest (expected one of: ",
         paste(header_probe_keys, collapse = ", "), ").")
  }
  
  man[]
}

# 4. Load Parquet β-matrix and identify CpG columns 
message("[INFO] Loading Parquet β-matrix from: ", INPUT_PARQUET)
tab <- arrow::read_parquet(INPUT_PARQUET)
dt  <- as.data.table(tab)
rm(tab); gc()

message("[INFO] Loaded matrix with shape: ", nrow(dt), " samples × ", ncol(dt), " columns")

# 4.1 Check that ID and label columns are present
meta_cols <- c(ID_COL, LABEL_COL)
if (!all(meta_cols %in% names(dt))) {
  stop("[ERROR] ID_COL and/or LABEL_COL not found in the Parquet file. ",
       "Available columns: ", paste(names(dt), collapse = ", "))
}

# 4.2 Identify CpG columns as "all columns except ID and label"
cpg_cols <- setdiff(names(dt), meta_cols)
message("[INFO] Detected ", length(cpg_cols), " CpG columns")

# 5. Load EPIC manifest and build Infinium Type I/II design vector 
message("[INFO] Loading EPIC manifest from: ", MANIFEST_PATH)
man_full <- load_illumina_manifest_any(MANIFEST_PATH)

if (!("CpG" %in% names(man_full))) {
  stop("[ERROR] Manifest did not contain a 'CpG' column after parsing.")
}

# 5.1 Detect the column that encodes Infinium probe design type
design_type_col <- NULL
for (cand in c("Infinium_Design_Type",
               "InfiniumDesignType",
               "Design_Type",
               "Infinium_Type",
               "Infinium.Design.Type")) {
  if (cand %in% names(man_full)) {
    design_type_col <- cand
    break
  }
}

if (is.null(design_type_col)) {
  stop("[ERROR] Could not find an Infinium design type column in manifest. ",
       "Looked for: Infinium_Design_Type, InfiniumDesignType, Design_Type, Infinium_Type, Infinium.Design.Type.")
}

message("[INFO] Using manifest columns: CpG id = 'CpG', design type = ", design_type_col)

# 5.2 Keep only manifest rows corresponding to CpGs present in the dataset
man_sub <- man_full[CpG %in% cpg_cols]
rm(man_full); gc()

message("[INFO] Manifest rows matching dataset CpGs: ", nrow(man_sub))
if (nrow(man_sub) == 0L) {
  stop("[ERROR] No CpGs from the manifest matched the dataset columns.")
}

# 5.3 Build raw design vector indexed by CpG ID
design_vec_raw <- man_sub[[design_type_col]]
names(design_vec_raw) <- man_sub[["CpG"]]
rm(man_sub); gc()

# 5.4 Align design vector to the order of CpG columns in the matrix
design_vec <- design_vec_raw[cpg_cols]
rm(design_vec_raw); gc()

# 5.5 Normalise to character "I" / "II"
design_vec <- as.character(design_vec)
design_vec[grepl("II", design_vec, ignore.case = TRUE)] <- "II"
design_vec[grepl("I",  design_vec, ignore.case = TRUE) &
             !is.na(design_vec) & design_vec != "II"] <- "I"

# 5.6 Drop CpGs with missing design type
if (any(is.na(design_vec))) {
  n_na <- sum(is.na(design_vec))
  message("[WARN] There are ", n_na, " CpGs without a design type; dropping them.")
  keep_idx   <- which(!is.na(design_vec))
  cpg_cols   <- cpg_cols[keep_idx]
  design_vec <- design_vec[keep_idx]
  message("[INFO] After dropping NA-design CpGs: ", length(cpg_cols), " CpGs remain.")
}

# 5.7 Convert "I"/"II" to integer 1/2 as expected by BMIQ
design_int <- integer(length(design_vec))
design_int[design_vec == "I"]  <- 1L
design_int[design_vec == "II"] <- 2L

if (any(design_int == 0L | is.na(design_int))) {
  stop("[ERROR] Some CpGs still have invalid design type after cleaning.")
}

# 5.8 Subset dt to metadata + CpGs with valid design type (memory friendly)
dt <- dt[, c(ID_COL, LABEL_COL, cpg_cols), with = FALSE]
gc()

# 6. Build β matrix and metadata vectors 
message("[INFO] Building β matrix (samples × CpGs) in memory...")
beta_mat <- as.matrix(dt[, ..cpg_cols])
rownames(beta_mat) <- dt[[ID_COL]]
storage.mode(beta_mat) <- "double"

id_tissue <- dt[[ID_COL]]
label     <- dt[[LABEL_COL]]
design_vec <- design_int

message("[INFO] β matrix dimensions: ", nrow(beta_mat), " samples × ",
        ncol(beta_mat), " CpGs")

# 7. Define safe per-sample BMIQ wrapper (sequential) 
# 7.1 Wrapper that checks minimum number of non-NA probes per type
bmiq_per_sample_safe <- function(b, design_vec, min_probes = 50L) {
  # 7.1.1 Ensure numeric vector
  b <- as.numeric(b)
  
  # 7.1.2 Sanity check on length
  if (length(b) != length(design_vec)) {
    stop("In bmiq_per_sample_safe: length(b) != length(design_vec)")
  }
  
  # 7.1.3 Split β-values by Infinium type (1 = Type I, 2 = Type II)
  beta1.v <- b[design_vec == 1L]
  beta2.v <- b[design_vec == 2L]
  
  n1 <- sum(!is.na(beta1.v))
  n2 <- sum(!is.na(beta2.v))
  
  # 7.1.4 If too few CpGs for one type, skip BMIQ and keep original values
  if (n1 < min_probes || n2 < min_probes) {
    warning(
      "Skipping BMIQ for this sample: not enough non-NA probes (Type I = ",
      n1, ", Type II = ", n2, ")"
    )
    return(b)
  }
  
  # 7.1.5 Run BMIQ
  res <- BMIQ(
    beta.v   = b,
    design.v = design_vec,
    plots    = FALSE
  )
  
  # 7.1.6 Return corrected β-values
  res$nbeta
}

# 8. Run sequential BMIQ correction in-place 
n_samples <- nrow(beta_mat)
message("[INFO] Starting BMIQ correction (sequential, in-place) on ", n_samples, " samples")

gc()  # Try to free memory before the loop

for (i in seq_len(n_samples)) {
  if (i == 1L || i %% 10L == 0L || i == n_samples) {
    message("[INFO] Sample ", i, "/", n_samples)
  }
  
  # 8.1 Extract β vector for sample i
  b <- beta_mat[i, ]
  
  # 8.2 Apply safe BMIQ and overwrite the row in-place
  beta_mat[i, ] <- bmiq_per_sample_safe(b, design_vec)
}

message("[INFO] BMIQ correction completed for all samples.")
gc()  # Free memory before building the final data.table

# 9. Reconstruct final data.table and write Parquet 
# 9.1 Build data.table from corrected β matrix
dt_bmiq <- as.data.table(beta_mat)

# 9.2 Attach ID and label columns
dt_bmiq[, (ID_COL) := id_tissue]
dt_bmiq[, (LABEL_COL) := label]

# 9.3 Reorder columns: ID | CpGs... | label
setcolorder(dt_bmiq, c(ID_COL, cpg_cols, LABEL_COL))

gc()  # One more GC before writing

# 9.4 Write BMIQ-corrected matrix to Parquet with LZ4 compression
message("[INFO] Writing BMIQ-corrected matrix to: ", OUTPUT_PARQUET)
arrow::write_parquet(
  dt_bmiq,
  sink        = OUTPUT_PARQUET,
  compression = "lz4"
)

message("[DONE] BMIQ-corrected dataset saved to: ", OUTPUT_PARQUET)
